# Notebook 01b - Data Augmentation

Load `health_data_clean.csv`, tao 5 bien the moi cho moi mau bang noise/scale/jitter, clip trong bien y te va luu `health_data_augmented.csv`.

In [ ]:
# Cell 1: Import va load data
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

np.random.seed(42)

DATA_CANDIDATES = [
    Path('health_data_clean.csv'),
    Path('../colab_export/health_data_clean.csv'),
    Path('ai_training/colab_export/health_data_clean.csv'),
    Path('/content/health_data_clean.csv'),
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Khong tim thay health_data_clean.csv. Hay upload file len Colab truoc.')

df = pd.read_csv(data_path)
print(f'Da load: {data_path}')
print(f'Dataset goc: {len(df)} mau')
print(df.head())

In [ ]:
# Cell 2: Neu health_data_clean.csv da bi scale 0-1 thi inverse ve don vi y te truoc khi augment
features = ['heart_rate','spo2','temperature','hr_rolling_mean',
            'hr_rolling_std','spo2_change','accel_mag']

missing = [c for c in features if c not in df.columns]
if missing:
    raise ValueError(f'Thieu cot bat buoc: {missing}')

looks_scaled = df[['heart_rate', 'spo2', 'temperature']].max().max() <= 1.5
if looks_scaled:
    scaler_candidates = [
        Path('scaler.pkl'),
        Path('../colab_export/scaler.pkl'),
        Path('ai_training/colab_export/scaler.pkl'),
        Path('/content/scaler.pkl'),
    ]
    scaler_path = next((p for p in scaler_candidates if p.exists()), None)
    if scaler_path is None:
        raise FileNotFoundError('health_data_clean.csv co ve da scale 0-1, can upload scaler.pkl de inverse_transform truoc khi augment.')
    scaler = joblib.load(scaler_path)
    df[features] = scaler.inverse_transform(df[features])
    print(f'Da inverse_transform bang {scaler_path}')
else:
    print('Du lieu dang o don vi y te, khong can inverse_transform')

In [ ]:
# Cell 3: Ham augment_sample(row, method)
def augment_sample(row, method):
    aug = row.copy()

    if method == 'noise':
        aug['heart_rate']  += np.random.normal(0, 1.5)
        aug['spo2']        += np.random.normal(0, 0.3)
        aug['temperature'] += np.random.normal(0, 0.05)
        if 'accel_mag' in aug:
            aug['accel_mag'] += np.random.normal(0, 0.02)

    elif method == 'scale':
        s = np.random.uniform(0.93, 1.07)
        aug['heart_rate']      *= s
        aug['hr_rolling_mean'] *= s
        aug['accel_mag']       *= s

    elif method == 'jitter':
        aug['hr_rolling_mean'] += np.random.normal(0, 2.0)
        aug['hr_rolling_std']  += np.random.normal(0, 0.5)
        aug['spo2_change']     += np.random.normal(0, 0.1)

    else:
        raise ValueError(f'Method khong hop le: {method}')

    aug['aug_method'] = method
    return aug


print('augment_sample OK')

In [ ]:
# Cell 4: Ap dung 5 lan moi mau -> dataset x5
methods = ['noise', 'scale', 'jitter', 'noise', 'scale']
augmented = []

for _, row in df.iterrows():
    for method in methods:
        augmented.append(augment_sample(row, method))

df_aug = pd.DataFrame(augmented).reset_index(drop=True)

# Clip trong bien y te
df_aug['heart_rate']      = df_aug['heart_rate'].clip(30, 220)
df_aug['spo2']            = df_aug['spo2'].clip(50, 100)
df_aug['temperature']     = df_aug['temperature'].clip(30, 43)
df_aug['hr_rolling_mean'] = df_aug['hr_rolling_mean'].clip(30, 220)
df_aug['hr_rolling_std']  = df_aug['hr_rolling_std'].clip(0, 80)
df_aug['spo2_change']     = df_aug['spo2_change'].clip(-50, 50)
df_aug['accel_mag']       = df_aug['accel_mag'].clip(0, 8)

df_aug.to_csv('health_data_augmented.csv', index=False)
print(f'{len(df)} mau -> {len(df_aug)} mau (x5)')
print('Da luu: health_data_augmented.csv')
print(df_aug.head())

In [ ]:
# Cell 5: Download tu Colab
try:
    from google.colab import files
    files.download('health_data_augmented.csv')
except Exception as exc:
    print('Khong phai Colab hoac khong download duoc:', exc)